# 02 · Live meekijken via MQTT — en een virtuele S31-CB

In notebook 01 zag je dat de Dragino S31-CB met `AT+PRO=3,5` zijn metingen als **JSON via MQTT** naar een broker stuurt. We hebben geen S31-CB in de klas, dus bouwen we er een na in Python die *exact hetzelfde* bericht publiceert. Iedereen in de klas kan zich op de broker abonneren en de "stal" live volgen.

## 1. Hoe MQTT werkt

* Een **broker** (server) ontvangt alle berichten en stuurt ze door. Toestellen praten nooit rechtstreeks met elkaar.
* Een client **publiceert** (`publish`) een bericht op een **topic**, bv. `ugent/i700301/stal/afdeling3`.
* Een andere client **abonneert** zich (`subscribe`) op een topic en krijgt vanaf dan elk bericht toegestuurd.
* **Wildcards:** `+` vervangt één niveau, `#` alles wat volgt. `ugent/i700301/stal/#` ontvangt alle afdelingen.
* **QoS** (Quality of Service): 0 = hoogstens één keer (fire and forget), 1 = minstens één keer, 2 = precies één keer. Een sensor op batterij gebruikt meestal 0 of 1.
* **Retain:** een bewaard bericht dat elke nieuwe abonnee meteen krijgt — handig voor "laatst gekende waarde".
* **Keepalive / Last Will:** de broker merkt wanneer een client wegvalt en kan dan een afscheidsbericht publiceren (bv. `status: offline`). Zo ontdek je een uitgevallen sensor.

Waarom is dit zo populair in IoT? De sensor hoeft maar één uitgaande verbinding te openen (geen poorten open te zetten), berichten zijn klein (2 bytes overhead), en wie de data wil (dashboard, klimaatcomputer, database) abonneert zich gewoon — de sensor hoeft daar niets van te weten.

## 2. Instellingen

> **Belangrijk — publieke broker.** `broker.hivemq.com` is voor iedereen open. Gebruik een **unieke klascode** (de docent geeft ze) zodat je niet met andere gebruikers botst, en publiceer nooit persoonlijke gegevens.

In [ ]:
import os
BROKER   = os.environ.get("IOT_BROKER", "broker.hivemq.com")
POORT    = int(os.environ.get("IOT_POORT", "1883"))
KLASCODE = os.environ.get("IOT_KLASCODE", "i700301-2026-xyz")   # <- docent: vervang door jullie eigen code
GROEP    = "groep01"                                             # <- jouw groepsnaam

BASIS = f"ugent/{KLASCODE}"
print("Topicprefix:", BASIS)

## 3. Abonneren: luister 20 seconden naar alles in de stal

Voer deze cel uit **terwijl de docent de virtuele sensor laat draaien** (sectie 5). Je ziet de berichten binnenkomen. Lukt poort 1883 niet, zie de websocket-tip in notebook 00.

In [ ]:
import paho.mqtt.client as mqtt
import time, json

def luister(topic, seconden=20, toon=True):
    '''Abonneer op `topic`, verzamel `seconden` lang berichten en geef ze terug als lijst.'''
    berichten = []
    def on_message(client, userdata, msg):
        berichten.append((msg.topic, msg.payload.decode(errors="replace"), msg.retain))
        if toon:
            print(f"[{time.strftime('%H:%M:%S')}] {msg.topic}  {msg.payload.decode(errors='replace')[:120]}")
    c = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    c.on_message = on_message
    c.connect(BROKER, POORT, keepalive=60)
    c.subscribe(topic, qos=1)
    c.loop_start(); time.sleep(seconden); c.loop_stop(); c.disconnect()
    print(f"\n{len(berichten)} bericht(en) ontvangen op '{topic}'.")
    return berichten

_ = luister(f"{BASIS}/stal/#", seconden=int(os.environ.get("IOT_LUISTER", "20")))

## 4. Zelf publiceren

Elke groep publiceert één bericht op zijn eigen topic. Kijk op het scherm vooraan (of in de cel van je buur) of het aankomt.

In [ ]:
c = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
c.connect(BROKER, POORT, keepalive=60)
c.loop_start()

bericht = {"groep": GROEP, "tekst": "hallo stal", "temperatuur": 21.5}
info = c.publish(f"{BASIS}/groepen/{GROEP}", json.dumps(bericht), qos=1)
info.wait_for_publish(timeout=5)
print("Gepubliceerd op", f"{BASIS}/groepen/{GROEP}", "->", bericht)

# Een 'retained' statusbericht: wie zich later abonneert, krijgt dit meteen
c.publish(f"{BASIS}/groepen/{GROEP}/status", "online", qos=1, retain=True).wait_for_publish(timeout=5)
c.loop_stop(); c.disconnect()

### Opdracht 4.1
Abonneer je met `luister(f"{BASIS}/groepen/+/status", 5)` op de statustopics van *alle* groepen. Wat merk je op over de berichten die je meteen binnenkrijgt, ook al publiceert er op dat moment niemand? (Kijk naar het derde element, `retain`.)

In [ ]:
# Opdracht 4.1 — controle
status = luister(f"{BASIS}/groepen/+/status", seconden=5)
print([(t, p, retain) for t, p, retain in status])

<details>
<summary><b>Oplossing tonen</b></summary>

De statusberichten komen **onmiddellijk** binnen, met `retain = True`: de broker bewaart per topic het laatste *retained* bericht en geeft het aan elke nieuwe abonnee. Zo weet een dashboard meteen welke sensoren "online" zijn zonder op hun volgende uplink te wachten. Een retained bericht blijft staan tot iemand een leeg retained bericht op hetzelfde topic publiceert.

</details>

## 5. De virtuele S31-CB (cel voor de docent — of voor wie zelf een "sensor" wil zijn)

Deze cel bootst drie Dragino S31-CB-sensoren na, één per stalafdeling. Ze publiceren het JSON-formaat uit de handleiding (sectie 2.2.1), met een licht variërende temperatuur en RV, en af en toe een alarm. In de echte sensor zou je dit bereiken met:

```
AT+PRO=3,5
AT+SERVADDR=broker.hivemq.com,1883
AT+CLIENT=s31cb-afdeling3
AT+PUBTOPIC=ugent/<klascode>/stal/afdeling3
AT+MQOS=1
AT+TDC=600
```

Het verschil: onze sensor zendt elke `INTERVAL` seconden i.p.v. elke 600, zodat je niet hoeft te wachten.

In [ ]:
import random, datetime as dt

INTERVAL = float(os.environ.get("IOT_INTERVAL", "5"))     # seconden tussen uplinks (echt: 600)
DUUR     = float(os.environ.get("IOT_DUUR", "180"))       # hoe lang de simulatie loopt (s)

afdelingen = {  # starttoestand per afdeling: temperatuur, RV, batterij
    "afdeling1": {"T": 22.0, "RV": 60.0, "bat": 3.52, "imei": "868508065628110"},
    "afdeling2": {"T": 23.5, "RV": 65.0, "bat": 3.41, "imei": "868508065628111"},
    "afdeling3": {"T": 27.8, "RV": 72.0, "bat": 3.29, "imei": "868508065628112"},
}

def s31cb_json(naam, s, historiek):
    nu = dt.datetime.now(dt.timezone.utc)
    bericht = {
        "IMEI": s["imei"], "IMSI": "206010000000000", "Model": "S31x-CB",
        "temperature": round(s["T"], 1), "humidity": round(s["RV"], 1),
        "interrupt": 0, "interrupt_level": 0, "count": 0,
        "battery": round(s["bat"], 2), "signal": random.randint(14, 26),
        "time": nu.strftime("%Y/%m/%d %H:%M:%S"),
        "latitude": 0.0, "longitude": 0.0, "gps_time": "1970-01-01T00:00:00Z",
    }
    for i, (T, RV, t) in enumerate(historiek[-8:][::-1], start=1):   # laatste 8 metingen, nieuwste eerst
        bericht[str(i)] = [T, RV, t]
    return bericht

sensor = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=f"s31cb-sim-{random.randint(0,9999)}")
sensor.connect(BROKER, POORT, keepalive=60)
sensor.loop_start()
hist = {n: [] for n in afdelingen}
start = time.time()
print(f"Virtuele sensoren publiceren {DUUR:.0f} s lang elke {INTERVAL:.0f} s op {BASIS}/stal/<afdeling> ...")
while time.time() - start < DUUR:
    for naam, s in afdelingen.items():
        s["T"]  = round(min(max(s["T"]  + random.gauss(0, 0.3), 15), 35), 1)
        s["RV"] = round(min(max(s["RV"] + random.gauss(0, 1.0), 30), 99), 1)
        s["bat"] = max(s["bat"] - 0.0005, 2.6)
        hist[naam].append([s["T"], s["RV"], dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")])
        bericht = s31cb_json(naam, s, hist[naam])
        if s["T"] >= 30 or s["RV"] >= 90:
            bericht["interrupt"] = 1                        # alarm: drempel overschreden
        sensor.publish(f"{BASIS}/stal/{naam}", json.dumps(bericht), qos=1)
    print(f"[{time.strftime('%H:%M:%S')}] " + "  ".join(f"{n}: {s['T']} °C / {s['RV']} %" for n, s in afdelingen.items()))
    time.sleep(INTERVAL)
sensor.loop_stop(); sensor.disconnect()
print("Simulatie gestopt.")

## 6. Studenten: de stal volgen en de data in een tabel zetten

Luister een minuut, zet de JSON-berichten om naar een `DataFrame` en teken het verloop per afdeling. Dit is wat een dashboard (Datacake, ThingsBoard, Node-RED) achter de schermen doet.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ruw = luister(f"{BASIS}/stal/+", seconden=int(os.environ.get("IOT_LUISTER2", "60")), toon=False)

rijen = []
for topic, payload, retain in ruw:
    try:
        b = json.loads(payload)
    except json.JSONDecodeError:
        continue                                   # geen JSON? overslaan
    rijen.append({"afdeling": topic.split("/")[-1], "tijd": pd.to_datetime(b["time"], format="%Y/%m/%d %H:%M:%S", utc=True),
                  "temperatuur_C": b["temperature"], "RV_pct": b["humidity"],
                  "batterij_V": b["battery"], "alarm": bool(b.get("interrupt", 0))})
df = pd.DataFrame(rijen)
print(df.groupby("afdeling")[["temperatuur_C", "RV_pct"]].agg(["mean", "min", "max"]).round(1) if len(df) else "Geen berichten ontvangen — draait de virtuele sensor?")
df.tail()

In [ ]:
if len(df):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for naam, g in df.groupby("afdeling"):
        ax.plot(g["tijd"], g["temperatuur_C"], marker=".", label=naam)
    ax.axhline(30, color="tab:red", linestyle="--", linewidth=1, label="alarmdrempel 30 °C")
    ax.set_ylabel("°C"); ax.set_title("Temperatuur per afdeling, live via MQTT"); ax.legend()
    fig.autofmt_xdate(); plt.tight_layout(); plt.show()

## 7. Zonder Python: dezelfde oefening in de browser

Wie geen notebook open heeft, kan meekijken met de **HiveMQ Websocket Client**: <https://www.hivemq.com/demos/websocket-client/>

1. Host `broker.hivemq.com`, Port `8884`, vink **SSL** aan, klik **Connect**.
2. Bij *Subscriptions* ▸ **Add New Topic Subscription**: `ugent/<klascode>/stal/#`.
3. Bij *Publish*: topic `ugent/<klascode>/groepen/<jouw groep>`, message `{"tekst": "hallo"}` ▸ **Publish**.

Poort 8884 is MQTT **over websockets met TLS** — dezelfde broker, een andere "deur". Zo kan een webpagina (die geen gewone TCP-verbindingen mag maken) toch met MQTT praten; dashboards in de browser doen dit allemaal.

## 8. Opdrachten

1. **Alarm.** Abonneer je op `{BASIS}/stal/#` en filter in Python de berichten met `"interrupt": 1`. Welke afdeling geeft alarm, en bij welke waarden?
2. **Last will.** Zoek in de documentatie van paho-mqtt (`client.will_set`) hoe je een afscheidsbericht instelt. Waarom is dat voor een stalsensor nuttiger dan voor een webwinkel?
3. **Koppeling met de klimaatcomputer.** Een klimaatcomputer die zich op `{BASIS}/stal/afdeling3` abonneert, kan de ventilatie bijsturen. Welke QoS zou je kiezen voor (a) de meting, (b) het alarm? Motiveer.
4. **Topic-ontwerp.** Ontwerp een topicstructuur voor een bedrijf met twee stallen, elk vier afdelingen, met per afdeling temperatuur, RV, CO₂ en NH₃. Hoe abonneer je je in één keer op alle NH₃-waarden?

In [ ]:
# Opdracht 1 — alarmen filteren
alarmen = [(t.split("/")[-1], json.loads(p)["temperature"], json.loads(p)["humidity"])
           for t, p, r in luister(f"{BASIS}/stal/#", seconden=30, toon=False)
           if json.loads(p).get("interrupt") == 1]
print(len(alarmen), "alarmberichten:", alarmen[:5])

<details>
<summary><b>Oplossing tonen</b></summary>

1. Meestal **afdeling3** (start op 27,8 °C, drempel 30 °C); de alarmvlag `interrupt` wordt 1 zodra T ≥ 30 °C of RV ≥ 90 %.
2. `client.will_set(topic, payload="offline", qos=1, retain=True)` vóór `connect()`. De broker publiceert dit zelf als de client wegvalt zonder nette `disconnect()`. Voor een stalsensor is dat cruciaal: een uitgevallen sensor (lege batterij, kabelbreuk, stroomuitval) wordt zo *actief* gemeld; bij een webwinkel is een verbroken verbinding van één gebruiker niet relevant.
3. Meting: QoS 0 of 1 (een gemiste meting is niet erg, over 10 min komt de volgende; QoS 0 bespaart batterij). Alarm: QoS 1 of 2 (mag nooit verloren gaan; QoS 2 vermijdt dubbele alarmen, maar kost twee extra berichten).
4. Bijvoorbeeld `bedrijf/stal<1-2>/afdeling<1-4>/<grootheid>` met grootheid `temperatuur`, `rv`, `co2`, `nh3`. Alle NH₃ in één keer: `bedrijf/+/+/nh3`. Alles van stal 2: `bedrijf/stal2/#`.

</details>